In [ ]:
import os
import cv2
import base64
import numpy as np
import json
import requests
import zipfile
from openai import OpenAI
from IPython.display import display, Image, Audio


api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print("Setup Complete. API Key configured.")

IMAGE_PATH = "medicin#2.jpg"

print("Setup Complete.")

Setup Complete.


In [11]:
# Task 1: Image Segmentation based on Template


REGIONS = {
    "Main Title": [339, 16, 498, 34],  
    "Price": [926, 74, 75, 36],
    "Product Picture": [34, 91, 266, 219],
    "Description": [350, 280, 650, 220],
    "How to use": [350, 555, 650, 180],
    "Ingredients": [350, 800, 650, 50],
    "Product Facts": [350, 940, 650, 295]
}

def crop_and_save_regions(image_path, regions):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not read image at {image_path}")
        return {}
    
    saved_paths = {}
    

    os.makedirs("cropped_regions", exist_ok=True)
    
    for name, (x, y, w, h) in regions.items():
        img_h, img_w = img.shape[:2]
        x = max(0, min(x, img_w))
        y = max(0, min(y, img_h))
        w = max(0, min(w, img_w - x))
        h = max(0, min(h, img_h - y))
        
        if w > 0 and h > 0:
            crop = img[y:y+h, x:x+w]
            filename = f"cropped_regions/{name.replace(' ', '_')}.jpg"
            cv2.imwrite(filename, crop)
            saved_paths[name] = filename
            print(f"Saved {name} to {filename}")
        else:
            print(f"Warning: Invalid crop dimensions for {name}")
            
    return saved_paths

# Execute extraction
region_files = crop_and_save_regions(IMAGE_PATH, REGIONS) 


Saved Main Title to cropped_regions/Main_Title.jpg
Saved Price to cropped_regions/Price.jpg
Saved Product Picture to cropped_regions/Product_Picture.jpg
Saved Description to cropped_regions/Description.jpg
Saved How to use to cropped_regions/How_to_use.jpg
Saved Ingredients to cropped_regions/Ingredients.jpg
Saved Product Facts to cropped_regions/Product_Facts.jpg


In [ ]:
# Encode Image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Call OpenAI Vision
def analyze_image_with_gpt4o(image_path, prompt_text):
    base64_image = encode_image(image_path)
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        },
                    },
                ],
            }
        ],
        max_tokens=300,
    )
    return response.choices[0].message.content

# Task 2: Obtain text from each region
extracted_data = {}
print("--- Task 2: Extracting Text from Regions ---")
for region_name, file_path in region_files.items():
    print(f"Analyzing {region_name}...")
    if region_name == "Product Picture": 
        text = analyze_image_with_gpt4o(file_path, "Describe this product image briefly.")
    else:
        text = analyze_image_with_gpt4o(file_path, "Transcribe the text and numbers in this image exactly as they appear.")
        
    extracted_data[region_name] = text
    print(f"[{region_name}]: {text[:100]}...")

# Task 3: Find title, Main color, special Info
print("\n--- Task 3: Global Analysis ---")
global_info_prompt = """
analyze this product image and identify:
1. The exact Main Title of the product.
2. The Main Color of the packaging.
3. Any Special Information (examples are warnings, certifications, 'prescription only').
Format as JSON.
"""
global_info = analyze_image_with_gpt4o(IMAGE_PATH, global_info_prompt)
print(global_info)

--- Task 2: Extracting Text from Regions ---
Analyzing Main Title...
[Main Title]: Alvedon, filmdragerad tablett 500 mg, 20 st...
Analyzing Price...
[Price]: 32 kr...
Analyzing Product Picture...
[Product Picture]: The image shows a package of "Alvedon 500 mg" film-coated tablets containing paracetamol. It's desig...
Analyzing Description...
[Description]: Alvedon, filmdragerad tablett 500 mg kan användas vid lindrig smärta (t.ex. huvudvärk) och som feber...
Analyzing How to use...
[How to use]: Observera! Högre doser än de rekommenderade innebär risk för mycket allvarlig leverskada. Alvedon sk...
Analyzing Ingredients...
[Ingredients]: Den aktiva substansen är paracetamol 500 mg. Övriga innehållsämnen är: Majsstärkelse, pregelatiniser...
Analyzing Product Facts...
[Product Facts]: EAN: 7046260976108

Varumärke: Alvedon

Kategori: Feber & Värk
Feber & Värk > Feber > Febernedsättan...

--- Task 3: Global Analysis ---
```json
{
    "Main Title": "Alvedon 500mg paracetamol",
    "Main Col

In [ ]:
# Task 4: Summary for blind person
def generate_accessibility_summary(extracted_info_dict, global_info_text):
    context = f"Global Info: {global_info_text}\n\nRegion Details:\n"
    for k, v in extracted_info_dict.items():
        context += f"{k}: {v}\n"
        
    prompt = f"""
    You are an assistant for a blind person shoping online. 
    Based on the following product details extracted from the package, provide a helpful, concise summary (max 300 words).
    Structure it logically: Product Name, what it is, Key ingredients/facts, usage instructions, and price.
    
    Data:
    {context}
    """
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print("\n--- Task 4: Summary ---")
summary_text = generate_accessibility_summary(extracted_data, global_info)
print(summary_text)

# Task 5: Talk-to-Speech 
# Generating speech for ssummary
print("\n--- Task 5: Generating Audio ---")
speech_file_path = "product_summary.mp3"
response = client.audio.speech.create(
  model="tts-1",
  voice="alloy",
  input=summary_text
)
response.stream_to_file(speech_file_path)
print(f"Audio saved to {speech_file_path}")

display(Audio(speech_file_path)) 




--- Task 4: Summary ---
Product Name: Alvedon 500mg Paracetamol

Description: Alvedon 500mg film-coated tablets provide temporary relief of mild pain (e.g., headaches) and reduce fever, such as during a cold. Not recommended for children under 3 years old. The pain-relief effect lasts around 4-5 hours.

Key Ingredients: Each tablet contains 500mg of paracetamol along with inactive ingredients such as corn starch, povidone, and more.

Usage Instructions: Adults and children over 40kg (from 12 years old) can take 1-2 tablets every 4-6 hours, up to 8 tablets per day. Children should be dosed based on weight. Do not exceed recommended doses without a doctor's approval.

Price: 32 kr

Alvedon is a non-prescription medication suitable for those with sensitive stomachs or increased bleeding tendencies. The packaging features a blue and white design with 20 tablets and a best before date of 2025-05-31. It is classified as a non-prescription drug under the Fever & Pain category.

--- Task 5: G

C:\Users\johnl\AppData\Local\Temp\ipykernel_7224\4197003515.py:35: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(speech_file_path)
